# Introduction
Experiments to see how Jupyter works with asyncio and external libraries

# Setup

## Imports

In [21]:
from nsepython import fnolist
import logging
from pathlib import Path
import os
import yaml
import sys
import pandas as pd
from ib_insync import Contract

## Set root directory

In [22]:
my_path = Path.cwd().parts

# find `ib` - the initial directory in the path
root = Path(*my_path[:my_path.index('ib')+1])
datapath = root / 'data'

# set `src` to be root path
if 'src' not in root.parts:
    root = root / 'src'

# set the os directory to root
os.chdir(root)
sys.path.append(str(root))

### Load modules from the new root

In [23]:
from support import Timer
from nse import make_chains

## Logging

In [24]:
# config log
with open(root.parent / 'config' / 'log.yml', 'r') as f:
    log_config = yaml.safe_load(f.read())
    logging.config.dictConfig(log_config)

log = logging.getLogger('log')

# Options

## Get the FNO list

In [25]:
symbols = fnolist()
df_syms = pd.DataFrame({'symbol': symbols}).assign(exchange='NSE')

## Get Option Chain Data

In [26]:
# Enable to regenerate option chains
opt_chain_timer = Timer('option chain')
opt_chain_timer.start()
df_chains = make_chains(df_syms=df_syms, savepath=datapath / 'nse_chains.json')
opt_chain_timer.stop()

2023-04-20 19:49:25,729 | root | option chain started at 20-Apr-2023 19:49:25


DIVISLAB:            100%|███████████████| 192/192 [03:48<00:00,  1.19s/it]


2023-04-20 19:53:14,776 | root | option chain took: 00:03:49 seconds


# History
## Get history of an equity

In [27]:
from nsepython import equity_history, equity_history_virgin
import datetime

symbol = 'SBIN'
series = 'EQ'
end_date = datetime.date(2023,4,6)
days_of_history = 365
intv = 50 # interval chunks

start_date = end_date - datetime.timedelta(days = days_of_history)

# determine start and end date for equity
eq_start = (end_date - datetime.timedelta(days=days_of_history)).strftime('%d-%m-%Y')
eq_end = end_date.strftime('%d-%m-%Y')

In [28]:
def dates_split(start_date: datetime.date, 
                end_date: datetime.date, 
                intv: int=50) -> tuple:
    
    """Splits dates into tuples of intervals"""

    if end_date < start_date:
        logging.error(f"End date:{end_date} cannot be earlier than Start date:{start_date} ")
        return None

    blks = ((end_date - start_date)/intv).days

    for _ in range(blks):
        end = start_date + datetime.timedelta(days=intv)
        yield (start_date, end)
        start_date = end + datetime.timedelta(days=1)
    
    # the last chunk
    if start_date < end_date:
        yield (start_date, end_date)



<hr>   

# WORK-IN-PROGRESS
[ ] fetch single history async   
[ ] fetch a block of history

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

# async def hist(symbol: str, start_date: datetime.date, end_date: datetime.date, series: str = 'EQ'):

#     h = equity_history_virgin(symbol = symbol, start_date=start_date, end_date=end_date, series=series)

#     return h

async def fetch_eq_hist(symbol: str, 
                        end_date: datetime.date, 
                        days_of_history: int=365, intv: int=50):
    
    class AsyncIter:
        """Makes iterable object blocks of contract lists"""    
        def __init__(self, items):    
            self.items = items    

        async def __aiter__(self):    
            for item in self.items:    
                yield item

    start_date = end_date - datetime.timedelta(days = days_of_history)

    tup_dates = dates_split(start_date, end_date)
    tup_str_dates = map(lambda x: (x[i].strftime('%d-%m-%Y') for i in x), tup_dates)

    async for tup in AsyncIter(tup_str_dates):

        # tasks = [asyncio.create_task(hist(symbol = symbol, series = 'EQ', start_date=tup[0], end_date=tup[1]))]
        tasks = [asyncio.create_task(equity_history_virgin(symbol = symbol, 
                                                           start_date=start.strftime('%d-%m-%Y'), 
                                                           end_date=end.strftime('%d-%m-%Y'), 
                                                           series='EQ'),
                                     name=start.strftime('%d-%m-%Y') & ':' & end.strftime('%d-%m-%Y')) 
                                        for start, end in tup_dates]

        results = await asyncio.gather(*tasks)

        return results
    

In [ ]:
asyncio.run(fetch_eq_hist(symbol=symbol, 
                        end_date=end_date, 
                        days_of_history=365, intv=50))

In [ ]:
symbol, series, eq_start, eq_end

In [ ]:
async def eq_hist(symbol, start_date, end_date):
    tasks = []

    class AsyncIter:
        """Makes iterable objects"""    
        def __init__(self, items):    
            self.items = items    

        async def __aiter__(self):    
            for item in self.items:    
                yield item
    
    async for eq_start, eq_end in AsyncIter(dates_split(start_date, end_date)):
        start = eq_start.strftime('%d-%m-%Y')
        end = eq_end.strftime('%d-%m-%Y')
        tasks.append[asyncio.create_task(equity_history_virgin(symbol, 'EQ', start, end), name=start + str((eq_end-eq_start).days))]

    result = await asyncio.gather(*tasks)
    return result

In [ ]:
r = asyncio.run(eq_hist(symbol, start_date, start_date + datetime.timedelta(days = 51)))

In [ ]:
equity_history_virgin(symbol, 'EQ', )

In [ ]:
tup_dates = dates_split(start_date, end_date)

In [ ]:
list(tup_dates)

In [ ]:

[(s.strftime('%d-%m-%Y'), e.strftime('%d-%m-%Y')) for s, e in tup_dates]

In [ ]:
list(map(lambda x: [x[i].strftime('%d-%m-%Y') for i in x], dates_split(start_date, end_date)))

In [ ]:
start_date = end_date - datetime.timedelta(days = days_of_history)
asyncio.run(hist(symbol=symbol, start_date=start_date, end_date=end_date))

In [ ]:
tup_dates = dates_split(start_date, end_date)
[d[0] for d in tup_dates]